In [ ]:
!pip install faiss-cpu tiktoken pypdf python-dotenv openai gradio --quiet
!pip install sentence-transformers --quiet


In [2]:
import os
import glob
import faiss
import numpy as np
from pypdf import PdfReader
from openai import OpenAI
import gradio as gr
import tiktoken

In [ ]:
os.environ["OPENAI_API_KEY"] = ""
client = OpenAI()

In [ ]:
def load_pdf(pdf_folder="/content"):
  documents = []
  filenames = []

  pdf_files = glob.glob(os.path.join(pdf_folder, "*.pdf"))
  print("found pdfs:", pdf_files)

  for pdf in pdf_files:
    reader = PdfReader(pdf)
    text = ""
    for page in reader.pages:
      text += page.extract_text() + "\n"

    documents.append(text)
    filenames.append(os.path.basename(pdf))

  return documents, filenames

docs, filenames = load_pdf()
print("Loaded", len(docs), "PDFs")


In [ ]:
def chunk_text(text, chunk_size=500, overlap=100):
  chunks = []
  start = 0
  while start < len(text):
    end = start + chunk_size
    chunk = text[start:end]
    chunks.append(chunk)
    start += chunk_size - overlap
  return chunks

all_chunks = []
chunk_sources = []

for doc, file in zip(docs, filenames):
  chunks = chunk_text(doc)
  all_chunks.extend(chunks)
  chunk_sources.extend([file] * len(chunks))

print("Total chunks: ", len(all_chunks))

In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")   # free & fast


In [7]:
def get_embeddings(text_list):
    return embed_model.encode(text_list, convert_to_numpy=True)


In [ ]:
embeddings = get_embeddings(all_chunks)
embeddings = np.array(embeddings).astype("float32")

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

In [ ]:
def retrieve(query, k=5):
  query_emb = np.array(get_embeddings([query])[0]).astype("float32")
  D, I = index.search(query_emb.reshape(1, -1), k)

  retrieved_chunks = [all_chunks[i] for i in I[0]]
  retrieved_sources = [chunk_sources[i] for i in I[0]]

  return retrieved_chunks, retrieved_sources

def generate_answer(query):
  chunks, sources = retrieve(query)

  context = ""
  for i, chunk in enumerate(chunks):
    context += f"source {i+1} ({sources[i]}):\n{chunk}\n\n"

  prompt = f""" you are a medical assistant. Answer the user's question strictly using

  CONTEXT:
  {context}

  QUESTION:
  {query}

  Provide a clear answer and mention sources at the end.
  """

  completion = client.chat.completions.create(
      model="gpt-4o-mini",
      messages=[{"role": "user", "content": prompt}],
  )

  return completion.choices[0].message.content

In [ ]:
def chat_fn(message, history):
  answer = generate_answer(message)
  return answer


ui = gr.ChatInterface(
    fn = chat_fn, title="Hospital AI Assistant",
    description="Ask health and hospital related questions"
)

ui.launch()